# CNT encoding ceiling (`app:ceiling`, supplement)

Cell 5: the CNT encode-decode round-trip ceiling -- KID/realism for {original, reconstruction, reconstruction-of-reconstruction, midpoint-of-two-reconstructions} against a native-128px-window reference (`ref_inc_win128.npy`, NOT the canonical 2,819-image reference), plus LPIPS and the plan-alignment Spearman rho (`CeilRho`).

**Needs:** DTD 120-pair subset, the CNT checkpoint.

**Maintained runnable path:** `experiments/ceiling_120.py` reproduces this; `score_only.py` rescores it (`app:ceiling-120`) from cached features. Kept for provenance.

# Two questions from a collaborator: (1) joint feature basis via side-by-side encoding, (2) how much does the CNT encoding itself limit the midpoint?

Same canonical setup cell as `finish_day` / `mechanism_study` / the correspondence notebook (CNT commit `0726930`, fp32). Outputs under `DATA_ROOT/joint_basis/`. Everything resumes from CSVs.

**Q1 — joint basis (cells 1–2).** CNT's splat and generator only accept square inputs, so the composite is a 512×512 image: A over its vertical mirror on the left, B over its mirror on the right (native texture scale, seamless continuation). The encoder runs at 512 with the same 100 slots, and each 256 quadrant is remapped into a standard 256-frame blob (`xy' = 2xy+3−6q`, `covs' = 4·covs`), after which the pipeline is *unchanged*: same `field(d,'ot',k)`, same decoder, same scoring. Three arms isolate the effect:

| arm | encoding | what it controls |
|---|---|---|
| `sep256` | A and B encoded separately at 256 (the paper's pipeline) | baseline |
| `self512` | A from `[A|A]` composite, B from `[B|B]` composite | same 512 regime and slot density, separate bases |
| `joint512` | A and B quadrants from the `[A|B]` composite | shared basis: slots, background feature and competition span both textures |

`joint512 − self512` is the basis effect; `self512 − sep256` is the price of the 512 regime. Cell 1 gates on reconstruction quality: if the 512 encodings can't reconstruct the endpoints, the arm is not interpretable and the strip-composite fallback (256 image, half-width strips) is the next thing to try.

**Q2 — encoding ceiling (cell 3).** Per image: PSNR / LPIPS / DINO distance of the encode→decode reconstruction, and of a second loop (recon of recon). Set level: KID and FID of originals, reconstructions, double reconstructions and the OT midpoints against the full DTD reference — the reconstruction rows are the ceiling any midpoint can reach. Then the decomposition per pair: quality(originals) → quality(recons) [encoding loss] → quality(midpoint) [interpolation loss], and whether endpoint reconstruction error predicts midpoint quality (Spearman, tertiles).


In [ ]:
# ============================================================
# 0. SETUP — canonical protocol (finish_day / mechanism cell 0, verbatim) + pinned CNT commit
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile,hashlib,itertools
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
assert torch.cuda.is_available(),'switch runtime to GPU'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'
MECH=f'{BASE}/mechanism'; os.makedirs(MECH,exist_ok=True); FIGS=f'{MECH}/figs'; os.makedirs(FIGS,exist_ok=True)
TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
t0=time.time()

# --- DTD ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system('rm -rf /content/_dtd'); os.system(f'unzip -qo "{ZIP}" -d /content/_dtd')
    inner=os.listdir('/content/_dtd'); src='/content/_dtd/images' if 'images' in inner else f'/content/_dtd/{inner[0]}'
    os.system(f'rm -rf {LOCAL} && mv "{src}" {LOCAL}')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 && cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
CNT_COMMIT=os.popen(f'cd {CNT} && git rev-parse HEAD').read().strip()
print(f'model ready {time.time()-t0:.0f}s | FMS={FMS} | CNT commit {CNT_COMMIT[:10]}')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def _pyr(si):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(FMS): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return sp
def build_style_maps(bf,si,fms):
    sp=_pyr(si); return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def decode_stagewise(blob_ref,sm_by_stage):
    """same decoder, but stage i receives sm_by_stage[i] (dict size->map). For E5 per-stage swaps."""
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm_by_stage[i][fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def sm_from_field(Z,L,H,W,Cc):
    it=Z.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
def _dec(d,Z,L,H,W,Cc,layout='a'): return decode_with_style_maps(d["blob_"+layout],sm_from_field(Z.float(),L,H,W,Cc))
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def pair_imgs(k): return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

# --- canonical Sinkhorn (as used in the paper) + converged log-domain Sinkhorn with duals ---
def sinkhorn_plan(x,y,eps,iters=ITERS):
    Cm=torch.cdist(x,y).pow(2); K=torch.exp(-Cm/eps)
    u=torch.ones(len(x),device=x.device)/len(x); v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u); b=torch.ones_like(v)
    for _ in range(iters): a=u/(K@b+1e-30); b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:], Cm
def sinkhorn_log(x,y,eps,iters=3000,tol=1e-6):
    """balanced, uniform marginals, float64 log-domain; returns P, duals f,g, cost, marginal residual, iterations."""
    X=x.double(); Y=y.double(); Cm=torch.cdist(X,Y).pow(2); n,m=Cm.shape
    la=-math.log(n); lb=-math.log(m); f=torch.zeros(n,device=X.device,dtype=torch.float64); g=torch.zeros(m,device=X.device,dtype=torch.float64)
    for it in range(1,iters+1):
        f=eps*(la-torch.logsumexp((g[None,:]-Cm)/eps,1)); g=eps*(lb-torch.logsumexp((f[:,None]-Cm)/eps,0))
        if it%10==0:
            logP=(f[:,None]+g[None,:]-Cm)/eps; res=float((torch.exp(logP).sum(1)-1/n).abs().max()*n)
            if res<tol: break
    logP=(f[:,None]+g[None,:]-Cm)/eps; P=torch.exp(logP)
    res=float(max((P.sum(1)-1/n).abs().max()*n,(P.sum(0)-1/m).abs().max()*m))
    return P,f,g,Cm,res,it
def plan_stats(P):
    P=P.double(); n=P.shape[0]; q=P/(P.sum(1,keepdim=True)+1e-300)
    Hrow=float(-(q*torch.log(q+1e-300)).sum(1).mean()); Htot=float(-(P*torch.log(P+1e-300)).sum())
    return dict(H_row=Hrow,H_total=Htot,MI_nats=math.log(n)-Hrow)
def natural_extension(x_all,y,g,eps,chunk=1024):
    """T(x)=sum_j q_j(x) y_j, q_j ∝ exp((g_j-||x-y_j||²)/eps) — dual-potential extension of a converged plan."""
    out=[]
    for i in range(0,len(x_all),chunk):
        Cm=torch.cdist(x_all[i:i+chunk].double(),y.double()).pow(2); q=torch.softmax((g[None,:]-Cm)/eps,1); out.append(q@y.double())
    return torch.cat(out)

# --- matching rules -> sampled targets (M,d); full field via canonical nn extension ---
def _targets(sf,tf,rule,seed,eps=EPS,M=NSUB,n_exact=1024,plan_kind='canonical'):
    seed=int(seed)   # numpy int64 from pandas breaks torch manual_seed
    N=sf.shape[0]; i_s=_sub(N,seed,'s')[:M]; i_t=_sub(N,seed,'t')[:M]; src,tgt=sf[i_s],tf[i_t]; info={}
    if rule in('ot','ot_sharp','ot_soft'):
        e={'ot':eps,'ot_sharp':0.005,'ot_soft':0.5}[rule] if rule!='ot' else eps
        if plan_kind=='canonical': P,_=sinkhorn_plan(src,tgt,e); info=dict(eps=e,**plan_stats(P))
        else: P,f,g,_,res,it=sinkhorn_log(src,tgt,e); info=dict(eps=e,res=res,iters=it,**plan_stats(P)); info['g']=g
        tt=((P/(P.sum(1,keepdim=True)+1e-30))@tgt.double()).float() if P.dtype==torch.float64 else (P/(P.sum(1,keepdim=True)+1e-30))@tgt
    elif rule=='exact':
        g_=torch.Generator(device=device).manual_seed(seed+11)
        js=torch.randperm(M,generator=g_,device=device)[:n_exact]; is_=torch.randperm(M,generator=g_,device=device)[:n_exact]
        r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
        tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
        ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]; info=dict(n_exact=n_exact)
    elif rule=='nn': tt=tgt[torch.cdist(src,tgt).argmin(1)]
    elif rule=='gauss':
        X,Y=src.double(),tgt.double(); mx,my=X.mean(0),Y.mean(0); Xc=X-mx; Yc=Y-my; Cc=X.shape[1]; I=torch.eye(Cc,device=device,dtype=torch.float64); ridge=1e-4
        Sa=(Xc.T@Xc)/(len(X)-1)+ridge*I; Sb=(Yc.T@Yc)/(len(Y)-1)+ridge*I
        def ps(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.sqrt())@V.T
        def pis(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.rsqrt())@V.T
        Sh=ps(Sa,ridge); Sih=pis(Sa,ridge); Amap=Sih@ps(Sh@Sb@Sh,0.0)@Sih; Amap=.5*(Amap+Amap.T)
        tt=(my+Xc@Amap.T).float(); info=dict(A=Amap,mu=(mx,my))
    elif rule=='uniform': tt=tgt.mean(0,keepdim=True).expand_as(src).clone()
    elif rule=='random':
        g_=torch.Generator(device=device).manual_seed(seed+7); tt=tgt[torch.randperm(M,generator=g_,device=device)]
    elif rule=='meanshift': tt=src+(tgt.mean(0)-src.mean(0))      # CNT-style native mean shift, adapted to the interpolation contract
    else: raise ValueError(rule)
    return i_s,tt,info
def field(d,rule,seed,**kw):
    seed=int(seed)
    """full-grid target field U (N,d) with the canonical nn extension; also returns X,Y grids and plan info."""
    L,Cc,H,W,sf,tf=_grids(d); i_s,tt,info=_targets(sf,tf,rule,seed,**kw)
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return dict(U=tt[nn],X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,i_s=i_s,tt=tt,nn=nn,info=info)
def zt(fd,t): return (1-t)*fd['X']+t*fd['U']
def decode_field(d,fd,Z): return _dec(d,Z,fd['L'],fd['H'],fd['W'],fd['Cc'])

# --- fixed-layout subspace of A: X = W_A F_A ---
def splat_W(d,side='a'):
    sp=_pyr(d['si_'+side]); Wm=sp[max(FMS)][0]; K=Wm.shape[0]
    return Wm.reshape(K,-1).T.contiguous().double(), d['bf_'+side][0].double()      # (N,K),(K,d)
def projector(Wm,tol=1e-6):
    Uw,S,_=torch.linalg.svd(Wm,full_matrices=False); keep=S>S[0]*tol; return Uw[:,keep], int(keep.sum())
def proj(Q,U): U=U.double(); return Q@(Q.T@U)
def e_sub(Q,U):
    U=U.double(); Uc=U-U.mean(0,keepdim=True); off=U-proj(Q,U)
    return dict(e_sub=float(off.norm()/(Uc.norm()+1e-12)),off_rms=float(off.norm()/math.sqrt(U.numel())),cen_rms=float(Uc.norm()/math.sqrt(U.numel())))
def e_aff(X,U,seed=0):
    """affine fit U≈XA+b on half the rows, residual on held-out rows."""
    X=X.double(); U=U.double(); N=len(X); g=torch.Generator(device=X.device).manual_seed(seed); p=torch.randperm(N,generator=g,device=X.device); tr,te=p[:N//2],p[N//2:]
    Xa=torch.cat([X,torch.ones(N,1,device=X.device,dtype=torch.float64)],1)
    G=Xa[tr].T@Xa[tr]; sol=torch.linalg.solve(G+1e-6*torch.trace(G)/G.shape[0]*torch.eye(G.shape[0],device=G.device,dtype=G.dtype),Xa[tr].T@U[tr]); R=U[te]-Xa[te]@sol; Uc=U[te]-U[te].mean(0,keepdim=True)
    return dict(e_aff=float(R.norm()/(Uc.norm()+1e-12)),aff_rms=float(R.norm()/math.sqrt(R.numel())))
def commutation_err(Q,Wm,Fm,Tfun):
    """J = T(W F) - W T(F), for a frozen map T applied to blob vectors (out-of-support warning if far from cloud)."""
    X=(Wm@Fm); TX=Tfun(X.float()).double(); TF=Tfun(Fm.float()).double(); J=TX-Wm@TF
    return dict(comm=float(J.norm()/(TX-TX.mean(0)).norm()),comm_rms=float(J.norm()/math.sqrt(J.numel())))
def neighbor_disc(U,H,W):
    G=U.reshape(H,W,-1); return float(((G[:,1:]-G[:,:-1]).norm(dim=-1).mean()+(G[1:]-G[:-1]).norm(dim=-1).mean())/2)
def var_ret(U,Y): return float(U.double().var(0).sum()/Y.double().var(0).sum())
def interaction_scale(X,Y):
    """RMS of -2 Xc Yc^T (centered interaction) vs raw median cost: the calibration the plan asks for."""
    Xc=X.double()-X.double().mean(0); Yc=Y.double()-Y.double().mean(0); M_=-2*Xc@Yc.T
    return dict(inter_rms=float(M_.pow(2).mean().sqrt()),cost_med=float(torch.cdist(X.double(),Y.double()).pow(2).median()))

# --- observers (canonical) ---
import timm, lpips as lpips_lib
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
lp=lpips_lib.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p:(torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
SCORE_RES=128
def at128(pil): return pil.convert('RGB').resize((SCORE_RES,SCORE_RES))
ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def kid_point(X,Y,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(len(X),len(Y)); x=X[:m]; y=Y[rng.choice(len(Y),m,replace=False)]
    kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
KS120=[int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k]; KSET=set(KS120)
DEV=[int(p_['k']) for p_ in MAN_ALL if int(p_['k']) not in KSET][:24]          # development set: pair-disjoint from the 120 confirmation pairs
print('dev pairs:',DEV); print('confirmation pairs:',len(KS120))

# --- per-pair scoring bundle for a decoded midpoint (human-study proxies) ---
_anc={}
def anchors(k):
    if k in _anc: return _anc[k]
    A,B=pair_imgs(k); rA=to_pil(make_recon(A)); rB=to_pil(make_recon(B)); f=dino_feats([at128(rA),at128(rB)])
    _anc[k]=dict(fA=f[0],fB=f[1],dAB=float(np.linalg.norm(f[0]-f[1])),rA=rA,rB=rB); return _anc[k]
def reencode_field(pil):
    e=cnt_load(pil); ba,bfa,sia=get_gen_inputs(e); sm=build_style_maps(bfa,sia,FMS); L=max(FMS); A=sm[L][0]; Cc=A.shape[0]
    return A.permute(1,2,0).reshape(-1,Cc)
def score_image(k,pil,Z=None,t=None):
    a=anchors(k); im=at128(pil); fi=inc_feats([im])[0]; fd=dino_feats([im])[0]
    r=dict(real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]),
           dA=float(np.linalg.norm(fd-a['fA'])/a['dAB']),dB=float(np.linalg.norm(fd-a['fB'])/a['dAB']))
    r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    if Z is not None:
        Xr=reencode_field(pil); r['roundtrip']=float((Xr.double()-Z.double()).norm()/(Z.double().norm()+1e-12))
    return r,fi,fd
_cache_d={}
def get_d(k):
    if k not in _cache_d:
        if len(_cache_d)>64: _cache_d.clear()
        A,B=pair_imgs(k); _cache_d[k]=build_d(cnt_load(A),cnt_load(B))
    return _cache_d[k]
def save_rows(rows,path): pd.DataFrame(rows).to_csv(path,index=False)
def load_rows(path,keys):
    if os.path.exists(path):
        df=pd.read_csv(path); return df.to_dict('records'), set(tuple(r[c] for c in keys) for r in df.to_dict('records'))
    return [],set()
print(f'setup done in {time.time()-t0:.0f}s')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False; torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
print('precision: fp32, deterministic cuDNN')


In [ ]:
for f in ['ref_inc_win128.npy','ref_dino_win128.npy','joint_basis.csv','feats.npz']:
    p=f'{BASE}/joint_basis/{f}'
    if os.path.exists(p): os.remove(p); print('removed',p)

In [ ]:
# ============================================================
# 1. HELPERS — the CNT encoder asserts a square 128/256 input, so the composite is a 256 image of two half-width centre strips:
#    [A_s | B_s] (A_s = centre 128x256 strip of A). Self-controls: [A_s | flipH(A_s)] and [flipH(B_s) | B_s] (same regime, separate bases).
#    OT runs between the two half-grids exactly as the paper's field() does (4096-sample canonical Sinkhorn + nn extension);
#    the decoded A-half is evaluated on its native 128x128 centre window, and sep256 is evaluated on the same window of the full-image output.
# ============================================================
from scipy import linalg as _sl
JB=f'{BASE}/joint_basis'; JIM=f'{JB}/imgs'; os.makedirs(JIM,exist_ok=True); HALF=IMG_SIZE//2; Q=IMG_SIZE//4
def tl(blob): return blob['layer_tree_levels'][-1][0]
def strip(img): return img.crop((Q,0,Q+HALF,IMG_SIZE))                                   # centre 128x256 strip
def composite(L,R): c=Image.new('RGB',(IMG_SIZE,IMG_SIZE)); c.paste(L,(0,0)); c.paste(R,(HALF,0)); return c
def flipH(im): return im.transpose(Image.FLIP_LEFT_RIGHT)
def win(img,half): x0=0 if half=='L' else HALF; return img.crop((x0,Q,x0+HALF,Q+HALF))     # native 128x128 window of one half of a 256 image
def win_full(img): return img.crop((Q,Q,Q+HALF,Q+HALF))                                 # the same texture region in a full-image (sep256) output
def enc(pil): return get_gen_inputs(cnt_load(pil))                                     # blob, bf, si
def style_full(bf,si): return build_style_maps(bf,si,FMS)[max(FMS)][0]                   # (C,256,256)
def half_cells(Sm,half): x=Sm[:,:,:HALF] if half=='L' else Sm[:,:,HALF:]; return x.permute(1,2,0).reshape(-1,x.shape[0])   # (256*128, C), row-major
def sm_replace(Sm,half,Z):
    it=Sm.clone(); blk=Z.reshape(IMG_SIZE,HALF,-1).permute(2,0,1).float()
    if half=='L': it[:,:,:HALF]=blk
    else: it[:,:,HALF:]=blk
    it=it.unsqueeze(0); return {s:(it if s==max(FMS) else F.interpolate(it,size=s,mode='bilinear',align_corners=False)) for s in set(FMS)}
def ot_U(sf,tf,seed):
    """canonical transported field on the source cells: 4096-sample Sinkhorn (eps=EPS) + nearest-sample extension, as in field()"""
    i_s,tt,info=_targets(sf,tf,'ot',seed); nn=torch.cdist(sf,sf[i_s]).argmin(1); return tt[nn],dict(i_s=i_s,info=info)
def n_active_in(blob,half):
    T=tl(blob); xy=T['xy'][0]; w=T['weights'][0,:,0]; return int(((w>0.5)&((xy[:,0]<0) if half=='L' else (xy[:,0]>=0))).sum())
def arms(k):
    A,B=pair_imgs(k); As,Bs=strip(A),strip(B); R={}
    d=get_d(k); R['sep256']=dict(kind='full',d=d,fd=field(d,'ot',k))
    bA,bfA,siA=enc(composite(As,flipH(As))); bB,bfB,siB=enc(composite(flipH(Bs),Bs)); SA,SB=style_full(bfA,siA),style_full(bfB,siB)
    R['self256']=dict(kind='half',blobA=bA,SA=SA,blobB=bB,SB=SB,X=half_cells(SA,'L'),Y=half_cells(SB,'R'),nA=n_active_in(bA,'L'),nB=n_active_in(bB,'R'))
    bJ,bfJ,siJ=enc(composite(As,Bs)); SJ=style_full(bfJ,siJ)
    R['joint256']=dict(kind='half',blobA=bJ,SA=SJ,blobB=bJ,SB=SJ,X=half_cells(SJ,'L'),Y=half_cells(SJ,'R'),nA=n_active_in(bJ,'L'),nB=n_active_in(bJ,'R'))
    return R,win_full(A),win_full(B)
def cloud_stats(X,Y,i_s,U,info):
    Xs=X[i_s].double(); Ys=Y[_sub(Y.shape[0],0,'t')].double(); cx=float(torch.cdist(Xs,Ys).pow(2).median()); wx=float(torch.cdist(Xs,Xs).pow(2).median()); wy=float(torch.cdist(Ys,Ys).pow(2).median())
    return dict(cost_med=cx,within_med=(wx+wy)/2,sep_ratio=cx/((wx+wy)/2+1e-12),mean_shift=float((Xs.mean(0)-Ys.mean(0)).norm()),var_ret=var_ret(U,Y),**{f'plan_{kk}':float(v) for kk,v in info.items() if isinstance(v,(int,float))})
def outputs(arm,k):
    """128x128 native windows: recA / mid / t1 on the A layout, recB on the B layout; plus cloud stats"""
    if arm['kind']=='full':
        d,fd=arm['d'],arm['fd']; dec=lambda Z: win_full(to_pil(decode_field(d,fd,Z)))
        o={'recA':dec(fd['X']),'mid':dec(zt(fd,0.5)),'t1':dec(fd['U']),'recB':win_full(to_pil(_dec(d,fd['Y'],fd['L'],fd['H'],fd['W'],fd['Cc'],layout='b')))}
        cs=cloud_stats(fd['X'],fd['Y'],fd['i_s'],fd['U'],fd['info']); cs.update(nA=int((tl(d['blob_a'])['weights']>0.5).sum()),nB=int((tl(d['blob_b'])['weights']>0.5).sum()))
    else:
        X,Y=arm['X'],arm['Y']; U,ex=ot_U(X,Y,k); dec=lambda Z: win(to_pil(decode_with_style_maps(arm['blobA'],sm_replace(arm['SA'],'L',Z))),'L')
        o={'recA':dec(X),'mid':dec(0.5*X+0.5*U),'t1':dec(U),'recB':win(to_pil(decode_with_style_maps(arm['blobB'],sm_replace(arm['SB'],'R',Y))),'R')}
        cs=cloud_stats(X,Y,ex['i_s'],U,ex['info']); cs.update(nA=arm['nA'],nB=arm['nB'])
    return o,cs
# --- reference features at the native 128 window scale (the canonical ref is 256->128 downsampled, i.e. a different scale) ---
def ref_window_feats():
    fp_i,fp_d=f'{JB}/ref_inc_win128.npy',f'{JB}/ref_dino_win128.npy'
    if os.path.exists(fp_i) and os.path.exists(fp_d): return np.load(fp_i),np.load(fp_d)
    pils=[]
    for f in sorted(glob.glob(f'{LOCAL}/*/*.jpg')):
        im=Image.open(f).convert('RGB'); w,h=im.size; r=IMG_SIZE/min(w,h); im=im.resize((max(IMG_SIZE,round(w*r)),max(IMG_SIZE,round(h*r))),Image.BICUBIC); w,h=im.size; l,t_=(w-IMG_SIZE)//2,(h-IMG_SIZE)//2
        crop=im.crop((l,t_,l+IMG_SIZE,t_+IMG_SIZE)).resize((SCORE_RES,SCORE_RES),Image.BICUBIC).resize((IMG_SIZE,IMG_SIZE),Image.BICUBIC); pils.append(win_full(crop))
    Ri=inc_feats(pils); Rd=dino_feats(pils); np.save(fp_i,Ri); np.save(fp_d,Rd); return Ri,Rd
REF_I,REF_D=ref_window_feats(); MANI_I=[Manifold(REF_I,seed=s) for s in range(5)]; MANI_D=[Manifold(REF_D,seed=s) for s in range(5)]
REF=REF_I.astype(np.float64); _kyy=((REF@REF.T/REF.shape[1]+1)**3); KYY=(_kyy.sum()-len(REF))/(len(REF)*(len(REF)-1)); del _kyy
def kid_ub(X,Y=REF,kyy=KYY):
    dd=X.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3; m=len(X); kxx=kk(X,X); return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+kyy-2*kk(X,Y).mean())
def fid(X,Y=REF):
    mx,my=X.mean(0),Y.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(Y,rowvar=False); cm=_sl.sqrtm(Sx@Sy); cm=cm.real if np.iscomplexobj(cm) else cm; return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*cm))
def bci(x,n=4000,seed=0):
    x=np.asarray(x,float); rng=np.random.RandomState(seed); v=x[rng.randint(len(x),size=(n,len(x)))].mean(1); return float(np.percentile(v,2.5)),float(np.percentile(v,97.5))
def psnr(p1,p2):
    a=np.asarray(p1.convert('RGB'),np.float32); b=np.asarray(p2.convert('RGB'),np.float32); mse=((a-b)**2).mean(); return float(10*np.log10(255**2/max(mse,1e-9)))
def score_win(pil,fA=None,fB=None):
    fi=inc_feats([pil])[0]; fdn=dino_feats([pil])[0]; r=dict(real_inc=float(realism(fi[None],MANI_I)[0]),real_dino=float(realism(fdn[None],MANI_D)[0]))
    if fA is not None: dAB=float(np.linalg.norm(fA-fB))+1e-9; r['dA']=float(np.linalg.norm(fdn-fA)/dAB); r['dB']=float(np.linalg.norm(fdn-fB)/dAB); r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    return r,fi,fdn
_pair_imgs_raw=pair_imgs
def pair_imgs(k):
    """what the canonical pipeline sees: every endpoint upsampled to IMG_SIZE (cnt_load does this internally)"""
    A,B=_pair_imgs_raw(k); return A.resize((IMG_SIZE,IMG_SIZE),Image.BICUBIC),B.resize((IMG_SIZE,IMG_SIZE),Image.BICUBIC)
print(f'reference windows: {len(REF_I)} | window = native {HALF}px, no resize')
# --- sanity on 5 pairs: reconstruction of the window per arm, round trip of the half-grid reshape, elements per half ---
S=[]
for k in KS120[:5]:
    R,wA,wB=arms(k); row=dict(k=k)
    Sm=R['joint256']['SA']; assert torch.equal(sm_replace(Sm,'L',half_cells(Sm,'L'))[max(FMS)][0],Sm),'half-grid reshape round trip failed'
    for arm in R:
        o,cs=outputs(R[arm],k); row[f'{arm}_lpipsA']=lpips_d(o['recA'],wA); row[f'{arm}_lpipsB']=lpips_d(o['recB'],wB); row[f'{arm}_psnrA']=psnr(o['recA'],wA); row[f'{arm}_nA']=cs['nA']; row[f'{arm}_sep']=cs['sep_ratio']
    S.append(row)
S=pd.DataFrame(S); print(S.round(3).to_string())
print('median window-recon LPIPS(A):',S[[c for c in S if c.endswith('lpipsA')]].median().round(3).to_dict(),'| active elements in the A half/image:',S[[c for c in S if c.endswith('_nA')]].median().to_dict())
GATE=float(S.self256_lpipsA.median())<=1.5*float(S.sep256_lpipsA.median())+0.05
print('GATE strip-composite reconstruction acceptable:',GATE,'-> if False the composite regime (half the slots per texture, seam) is the confound, not the basis')

In [ ]:
# ============================================================
# 2. Q1 — joint basis on the 120 pairs: recA / recB / mid / t1 windows per arm, cloud statistics, native-window scoring, KID with paired ΔKID
# ============================================================
CSVJ=f'{JB}/joint_basis.csv'; rows,have=load_rows(CSVJ,['k','arm','what']); FE=dict(np.load(f'{JB}/feats.npz')) if os.path.exists(f'{JB}/feats.npz') else {}
ARMS=['sep256','self256','joint256']; WHATS=['recA','recB','mid','t1']; t0=time.time()
for n_,k in enumerate(KS120):
    if all((k,a,w) in have for a in ARMS for w in WHATS): continue
    R,wA,wB=arms(k); fA=dino_feats([wA])[0]; fB=dino_feats([wB])[0]
    for arm in ARMS:
        o,cs=outputs(R[arm],k)
        for w,pil in o.items():
            if (k,arm,w) in have: continue
            os.makedirs(f'{JIM}/{arm}_{w}',exist_ok=True); pil.save(f'{JIM}/{arm}_{w}/{k:04d}.png'); s,fi,fdn=score_win(pil,fA,fB); FE[f'{arm}|{w}|{k}|inc']=fi; FE[f'{arm}|{w}|{k}|dino']=fdn
            rows.append(dict(k=k,arm=arm,what=w,**cs,**s,lpips_A=lpips_d(pil,wA),lpips_B=lpips_d(pil,wB),psnr_A=psnr(pil,wA),psnr_B=psnr(pil,wB)))
    if n_%10==0: save_rows(rows,CSVJ); np.savez(f'{JB}/feats.npz',**FE); print(f'{n_+1}/120 {time.time()-t0:.0f}s')
save_rows(rows,CSVJ); np.savez(f'{JB}/feats.npz',**FE); J=pd.DataFrame(rows); ks=sorted(set(J.k))
X={(a,w):np.stack([FE[f'{a}|{w}|{k}|inc'] for k in ks]).astype(np.float64) for a in ARMS for w in WHATS}
rng=np.random.RandomState(0); draws={key:[] for key in X}
for _ in range(200):
    idx=rng.choice(len(ks),100,replace=False)
    for key in X: draws[key].append(kid_ub(X[key][idx]))
KID={f'{a}|{w}':dict(KID=kid_ub(X[(a,w)]),FID=fid(X[(a,w)])) for a,w in X}
def _dk(a,b,w,name):
    dv=np.array(draws[(a,w)])-np.array(draws[(b,w)]); KID[f'{a}|{w}'][name]=KID[f'{a}|{w}']['KID']-KID[f'{b}|{w}']['KID']; KID[f'{a}|{w}'][name+'_lo'],KID[f'{a}|{w}'][name+'_hi']=float(np.percentile(dv,2.5)),float(np.percentile(dv,97.5))
for w in WHATS: _dk('self256','sep256',w,'d_vs_sep256'); _dk('joint256','sep256',w,'d_vs_sep256'); _dk('joint256','self256',w,'d_vs_self256')
KT=pd.DataFrame(KID).T; print('=== set level, native 128 windows vs DTD window reference (n=%d pairs) ==='%len(ks)); print(KT.round(4).to_string())
print('\n=== per-pair medians ==='); print(J.pivot_table(index=['what','arm'],values=['real_inc','real_dino','place','lpips_A','lpips_B','psnr_A','cost_med','sep_ratio','mean_shift','var_ret','nA'],aggfunc='median').round(3).to_string())
M=J[J.what=='mid'].pivot(index='k',columns='arm'); PP={}
for o in ['real_inc','real_dino','lpips_A','lpips_B','cost_med','sep_ratio','mean_shift','var_ret']:
    for a,b in [('joint256','self256'),('self256','sep256'),('joint256','sep256')]:
        x=(M[o][a]-M[o][b]).dropna().values; lo,hi=bci(x); PP[f'{o}|{a}-{b}']=dict(mean=float(x.mean()),lo=lo,hi=hi,p=float(wilcoxon(x).pvalue) if np.abs(x).sum()>0 else 1.0)
print('\n=== midpoint paired contrasts (joint−self = basis effect; self−sep = composite-regime price) ==='); print(pd.DataFrame(PP).T.round(4).to_string())
json.dump(dict(kid=KID,paired=PP,n=len(ks)),open(f'{JB}/joint_summary.json','w'),indent=1)

In [ ]:
# ============================================================
# 3. Q2 — encoding ceiling: recon and double-recon quality per image; set-level ceiling table; encoding vs interpolation loss; does recon error predict midpoint quality?
# ============================================================
REFC=ref_inc.astype(np.float64); _k=((REFC@REFC.T/REFC.shape[1]+1)**3); KYYC=(_k.sum()-len(REFC))/(len(REFC)*(len(REFC)-1)); del _k
kidc=lambda X: kid_ub(X,REFC,KYYC); fidc=lambda X: fid(X,REFC)
CSVC=f'{JB}/ceiling.csv'; rows,have=load_rows(CSVC,['k','side','what']); FC=dict(np.load(f'{JB}/feats_ceiling.npz')) if os.path.exists(f'{JB}/feats_ceiling.npz') else {}; t0=time.time()
def recon_of(pil): return to_pil(make_recon(pil))
for n_,k in enumerate(KS120):
    if all((k,s_,w) in have for s_ in 'AB' for w in ['orig','rec1','rec2']): continue
    A,B=pair_imgs(k)
    for s_,img in [('A',A),('B',B)]:
        r1=recon_of(img); r2=recon_of(r1)
        for w,pil in [('orig',img),('rec1',r1),('rec2',r2)]:
            if (k,s_,w) in have: continue
            im=at128(pil); fi=inc_feats([im])[0]; fdn=dino_feats([im])[0]; FC[f'{s_}|{w}|{k}|inc']=fi; FC[f'{s_}|{w}|{k}|dino']=fdn
            rows.append(dict(k=k,side=s_,what=w,real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fdn[None],MANI_DINO)[0]),psnr_orig=psnr(pil,img),lpips_orig=lpips_d(pil,img),
                             dino_orig=float(np.linalg.norm(fdn-dino_feats([at128(img)])[0])),lpips_prev=(0.0 if w=='orig' else lpips_d(pil,img if w=='rec1' else r1))))
    if n_%20==0: save_rows(rows,CSVC); np.savez(f'{JB}/feats_ceiling.npz',**FC); print(f'{n_+1}/120 {time.time()-t0:.0f}s')
save_rows(rows,CSVC); np.savez(f'{JB}/feats_ceiling.npz',**FC); C=pd.DataFrame(rows); ks=sorted(set(C.k))
# --- set-level ceiling: originals / rec1 / rec2 (A∪B, 240 images) and the midpoints (paper's ot dir if present, else this notebook's sep256 mid) ---
def stack(w): return np.stack([FC[f'{s_}|{w}|{k}|inc'] for k in ks for s_ in 'AB']).astype(np.float64)
CE={w:dict(KID=kid_ub(stack(w)),FID=fid(stack(w)),n=2*len(ks)) for w in ['orig','rec1','rec2']}
midsrc=f'{SCALE}/ot' if os.path.isdir(f'{SCALE}/ot') else f'{JIM}/sep256_mid'; mids=[Image.open(f'{midsrc}/{k:04d}.png').convert('RGB') for k in ks if os.path.exists(f'{midsrc}/{k:04d}.png')]
if mids:
    Xm=inc_feats([at128(p) for p in mids]).astype(np.float64); CE['midpoint_ot']=dict(KID=kid_ub(Xm),FID=fid(Xm),n=len(mids)); Fm=dino_feats([at128(p) for p in mids]); midreal=dict(zip([k for k in ks if os.path.exists(f'{midsrc}/{k:04d}.png')],realism(Fm,MANI_DINO)))
print('=== ceiling (vs full DTD reference; KID at n≈240 originals / ≈120 midpoints — compare rows, not to paper tables) ==='); print(pd.DataFrame(CE).T.round(4).to_string())
print('\n=== per-image reconstruction (median): rec1 = one encode/decode loop, rec2 = two ==='); print(C.pivot_table(index='what',values=['psnr_orig','lpips_orig','dino_orig','real_inc','real_dino','lpips_prev'],aggfunc='median').round(3).to_string())
# --- decomposition: encoding loss vs interpolation loss (DINO realism, per pair) ---
P_=C.pivot_table(index='k',columns=['what','side'],values='real_dino'); R1=C.pivot_table(index='k',columns=['what','side'],values='lpips_orig')['rec1']
if mids:
    dec=pd.DataFrame({'orig':P_['orig'].mean(1),'rec1':P_['rec1'].mean(1),'mid':pd.Series(midreal)}).dropna(); dec['encoding_loss']=dec.orig-dec.rec1; dec['interp_loss']=dec.rec1-dec.mid
    print('\n=== realism decomposition (DINO realism, mean of A,B endpoints): originals -> recons -> OT midpoint ==='); print(dec[['orig','rec1','mid','encoding_loss','interp_loss']].describe().loc[['mean','50%','25%','75%']].round(4).to_string())
    x=(R1.max(1)).loc[dec.index]; rho=spearmanr(x,dec.mid); rho2=spearmanr(x,dec.interp_loss)
    print(f"\nSpearman(worst endpoint recon LPIPS, midpoint realism) = {rho.correlation:+.3f} (p={rho.pvalue:.3g}) | vs interpolation loss = {rho2.correlation:+.3f} (p={rho2.pvalue:.3g})")
    ter=pd.qcut(x,3,labels=['good recon','mid','bad recon'])
    tab=[]
    for lab in ['good recon','mid','bad recon']:
        kk=[k for k in dec.index if ter.loc[k]==lab]; Xm_=inc_feats([at128(Image.open(f'{midsrc}/{k:04d}.png').convert('RGB')) for k in kk]).astype(np.float64)
        tab.append(dict(tertile=lab,n=len(kk),recon_lpips_max=float(x.loc[kk].median()),mid_realism=float(dec.mid.loc[kk].median()),mid_KID=kid_ub(Xm_),rec_KID=kid_ub(np.stack([FC[f'{s_}|rec1|{k}|inc'] for k in kk for s_ in 'AB']).astype(np.float64))))
    print('\n=== midpoint quality by endpoint-reconstruction tertile (KID at n≈40 is noisy; read the trend) ==='); print(pd.DataFrame(tab).round(4).to_string())
    json.dump(dict(ceiling=CE,decomposition=dec.describe().to_dict(),spearman=dict(rho_mid=float(rho.correlation),p_mid=float(rho.pvalue),rho_interp=float(rho2.correlation),p_interp=float(rho2.pvalue)),tertiles=tab),open(f'{JB}/ceiling_summary.json','w'),indent=1,default=float)


In [ ]:
sz=[(k,)+Image.open(f'{SCALE}/A/{k:04d}.png').size+Image.open(f'{SCALE}/B/{k:04d}.png').size for k in KS120]
bad=[s for s in sz if s[1:]!=(256,256,256,256)]; print(len(bad),'pairs not 256x256:',bad[:10])

In [ ]:
# ============================================================
# 4. FIGURES — arm comparison, per-pair basis effect, recon-error vs midpoint-quality, qualitative grid
# ============================================================
FIGD=f'{JB}/figs'; os.makedirs(FIGD,exist_ok=True)
fig,ax=plt.subplots(1,3,figsize=(15,4))
kt=pd.DataFrame(KID).T; labs=[f'{a}|{w}' for w in ['recA','mid','t1'] for a in ARMS]; ax[0].bar(range(len(labs)),[kt.loc[l,'KID'] for l in labs]); ax[0].set_xticks(range(len(labs))); ax[0].set_xticklabels(labs,rotation=75,fontsize=7); ax[0].set_ylabel('KID vs window reference'); ax[0].set_title('Q1: encoding arms')
Mm=J[J.what=='mid'].pivot(index='k',columns='arm',values='real_dino'); ax[1].scatter(Mm['self256'],Mm['joint256'],s=10); lim=[Mm.min().min(),Mm.max().max()]; ax[1].plot(lim,lim,'k--',lw=.8); ax[1].set_xlabel('midpoint realism, separate bases'); ax[1].set_ylabel('midpoint realism, joint basis'); ax[1].set_title('basis effect per pair')
if 'dec' in globals(): ax[2].scatter(x,dec.mid,s=10); ax[2].set_xlabel('worst endpoint recon LPIPS'); ax[2].set_ylabel('OT midpoint DINO realism'); ax[2].set_title(f'Q2: ρ={rho.correlation:+.2f}')
plt.tight_layout(); plt.savefig(f'{FIGD}/joint_and_ceiling.png',dpi=160); plt.show()
show=sorted(KS120)[:6]; cols=[('rec A',f'{JIM}/sep256_recA'),('mid sep256',f'{JIM}/sep256_mid'),('rec A self',f'{JIM}/self256_recA'),('mid self256',f'{JIM}/self256_mid'),('rec A joint',f'{JIM}/joint256_recA'),('mid joint256',f'{JIM}/joint256_mid'),('rec B',f'{JIM}/sep256_recB')]
fig,ax=plt.subplots(len(show),len(cols),figsize=(1.7*len(cols),1.7*len(show)))
for i,k in enumerate(show):
    for j,(t_,p) in enumerate(cols):
        fp=f'{p}/{k:04d}.png'; ax[i,j].axis('off')
        if os.path.exists(fp): ax[i,j].imshow(Image.open(fp).convert('RGB'))
        if i==0: ax[i,j].set_title(t_,fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIGD}/qualitative.png',dpi=160,bbox_inches='tight'); plt.show(); print('figures ->',FIGD)

In [ ]:
# ============================================================
# 5. Which textures reconstruct best / worst after one encode→decode loop (from cell 3's ceiling.csv)
# ============================================================
C=pd.read_csv(f'{JB}/ceiling.csv'); PINFO={int(p_['k']):p_ for p_ in MAN_ALL}
def cat_of(k,side):
    p=PINFO[k]
    for key in (['catA','cat_a','category_a'] if side=='A' else ['catB','cat_b','category_b']):
        if key in p: return str(p[key])
    f=str(p.get('fA' if side=='A' else 'fB','')); return os.path.basename(os.path.dirname(f)) or '?'
r1=C[C.what=='rec1'].copy(); r1['category']=[cat_of(k,s) for k,s in zip(r1.k,r1.side)]
orig_real=C[C.what=='orig'].set_index(['k','side']).real_dino
r1['real_drop']=[float(orig_real.loc[(k,s)])-v for k,s,v in zip(r1.k,r1.side,r1.real_dino)]
rank=r1.sort_values('lpips_orig')[['k','side','category','psnr_orig','lpips_orig','dino_orig','real_dino','real_drop']].reset_index(drop=True); rank.to_csv(f'{JB}/recon_ranking.csv',index=False)
N=8
print('=== BEST reconstructed (lowest LPIPS to original) ==='); print(rank.head(N).round(3).to_string())
print('\n=== WORST reconstructed (highest LPIPS) ==='); print(rank.tail(N).iloc[::-1].round(3).to_string())
g=r1.groupby('category').lpips_orig.agg(['median','count']); g=g[g['count']>=3].sort_values('median')
if len(g)>1: print('\nby category (median LPIPS, n≥3):'); print(g.round(3).to_string())
else: print('\n(no category labels in the manifest — keys are:',list(PINFO[KS120[0]].keys()),')')
print('\nrank agreement between metrics (Spearman): LPIPS vs PSNR %.2f | LPIPS vs DINO dist %.2f | LPIPS vs realism drop %.2f'%(spearmanr(r1.lpips_orig,-r1.psnr_orig).correlation,spearmanr(r1.lpips_orig,r1.dino_orig).correlation,spearmanr(r1.lpips_orig,r1.real_drop).correlation))
# --- grid: original | recon | recon of recon, for the N best (top block) and N worst (bottom block) ---
def orig_of(k,side): A,B=pair_imgs(k); return A if side=='A' else B
sel=[('best',r) for _,r in rank.head(N).iterrows()]+[('worst',r) for _,r in rank.tail(N).iloc[::-1].iterrows()]
fig,ax=plt.subplots(len(sel),3,figsize=(6.3,2.1*len(sel)))
for i,(lab,r) in enumerate(sel):
    k,s=int(r['k']),r['side']; o=orig_of(k,s); rc=to_pil(make_recon(o)); rc2=to_pil(make_recon(rc))
    titles=[f'{lab} #{i%N+1}: k={k}{s} {r["category"]}',f'recon  LPIPS {r["lpips_orig"]:.2f}  PSNR {r["psnr_orig"]:.1f}','recon of recon']
    for j,(t_,im) in enumerate(zip(titles,[o,rc,rc2])):
        ax[i,j].imshow(im); ax[i,j].axis('off'); ax[i,j].set_title(t_,fontsize=7)
plt.tight_layout(); os.makedirs(f'{JB}/figs',exist_ok=True); plt.savefig(f'{JB}/figs/recon_best_worst.png',dpi=150,bbox_inches='tight'); plt.show(); print('ranking ->',f'{JB}/recon_ranking.csv')